# Hierarchical Models

`caskade` is a very general simulator building framework, this includes hierarchical models. A hierarchical model is essentially a model within a model, and this can take several forms. Here we will demonstrate the basic patterns of hierarchical models that can be constructed in `caskade`, from which you should be able to build any simulator.

In [ ]:
import caskade as ck
import torch
import matplotlib.pyplot as plt
import numpy as np


class Gaussian(ck.Module):
    def __init__(self, name, x0=None, q=None, phi=None, sigma=None, I0=None):
        super().__init__(name)
        self.x0 = ck.Param("x0", x0, shape=(2,))  # position
        self.q = ck.Param("q", q, shape=())  # axis ratio
        self.phi = ck.Param("phi", phi, shape=())  # orientation
        self.sigma = ck.Param("sigma", sigma, shape=())  # width
        self.I0 = ck.Param("I0", I0, shape=())  # intensity

    @ck.forward
    def _r(self, x, y, x0=None, q=None, phi=None):
        x, y = x - x0[..., 0], y - x0[..., 1]
        s, c = torch.sin(phi), torch.cos(phi)
        x, y = c * x - s * y, s * x + c * y
        return (x**2 + (y * q) ** 2).sqrt()

    @ck.forward
    def brightness(self, x, y, sigma=None, I0=None):
        return I0 * (-self._r(x, y) ** 2 / sigma**2).exp()


X, Y = torch.meshgrid(torch.linspace(-5, 5, 100), torch.linspace(-5, 5, 100), indexing="xy")

## Set param values during simulation (live params)

Sometimes you want your simulator to decide, within the simulation, what value a param should have. This is the simplest hierarchical model and can be created using a `static` param with a value of `None`, typically called a `Live param`. By being a `static` param, `caskade` will not expect the value as an input to the simulation, the `None` value tells `caskade` that it will be set during the simulation (anything else will mean `caskade` throws an error if you try to do this).

If this all sounds a lot like a `pointer` param, which `caskade` already has, you are right. Functionally, these two capabilities are nearly identical. As a rule of thumb, `pointer` params should be kept very light, either a direct pointer or a simple single (lambda) function to translate/combine a few params, while `live` params can do more heavy lifting as one can make use of the functions/utilities in the simulator quite easily (and more readably).

In [ ]:
class SetLiveParam(ck.Module):
    def __init__(self, gaussian: Gaussian, a: float = 1.0, b: float = 2.0):
        super().__init__()
        self.gaussian = gaussian
        self.gaussian.sigma.to_static(None)  # Live param
        self.a = ck.Param("a", a, dynamic=True)
        self.b = ck.Param("b", b, dynamic=False)
        self.c = ck.Param("c", None, dynamic=False)  # Live param

    @ck.forward
    def determine_c(self, a, b):  # cannot request c here
        self.c = a + b

    @ck.forward
    def determine_sigma(self, a, b, c):
        self.gaussian.sigma = (a + b + c) / 2

    @ck.forward
    def main_sim(self, x, y, a, b):  # cannot request c here

        self.determine_c()

        self.determine_sigma()

        print("within sim c =", self.c.value.item())  # can manually get c after setting value

        return self.gaussian.brightness(x, y) * (a + b)


G = Gaussian("G", x0=[0, 0], q=1, phi=0, I0=1.0)
sim = SetLiveParam(G)
sim.graphviz()

Notice that the live params are greyed out like other `static` params, but also have a black border like `dynamic` params since they are somewhere in between the two.

In [ ]:
plt.imshow(sim.main_sim(X, Y, sim.get_values()))
plt.axis("off")
plt.show()
print("c is cleared after simulation (should be None):", sim.c.value)
print("Gaussian sigma is cleared after simulation (should be None):", sim.gaussian.sigma.value)

## Batching a subgraph (hierarchical graphical models)

The goal of `caskade` is to support modular development of code. A quite common pattern is to develop a simulator to handle a single case, then use `vmap` to batch over many cases. This is great design and makes for very readable implementations that behave intuitively. All is well until you want that batched simulator to operate inside another simulator (see this real world example in `caustics`). Since `caksade` handles moving param values around for you, the inner simulator is never handed its values, so there is nothing to `vmap` over. This is where `caskade`'s hierarchical linking comes into play. A hierarchical link says that the child is not a part of the parent's simulation. It is a simulator of its own, which the parent runs as many times as it likes with values that the parent hands to it.

In this example, lets use our gaussian blob. The code is designed to create a single gaussian blob, but with a batched simulator we can model multiple simultaneously. So now we will create an image with multiple blobs (similarly, one could model multiple images with a different blob in each).

In [ ]:
class MultiBlob(ck.Module):
    def __init__(self, gaussian):
        super().__init__()
        self.hierarchical_link(
            "gauss", gaussian
        )  # note, the name string is how we will reference the module

    @ck.forward
    def brightness(self, x, y):
        params = self.gauss.get_values()  # one row of params for each blob
        batch_gauss = torch.vmap(lambda p: self.gauss.brightness(x, y, params=p))
        return batch_gauss(params).sum(dim=0)


G = Gaussian("G", q=1.0, phi=0.0, sigma=1.0, I0=1.0)
G.x0 = torch.tensor([[0, 0], [3, 1], [-2, 4], [-3, -3], [1, -4]])  # Batched param
print(f"x0 shape: {G.x0.shape}, x0 batched: {G.x0.batched}, x0 batch shape: {G.x0.batch_shape}")
Blobs = MultiBlob(G)
Blobs.graphviz()

Notice that there is a box around the part of the graph that is getting batched. This is a common notation for probabilistic graphical models to show that a subsection of the graph is repeated (batched) many times.

### How does `MultiBlob` run the blobs?

It uses the same two calls you would use on any simulator from the outside, only from inside a running simulation.

- `self.gauss.get_values()` returns the current values of the dynamic params in the box, as the module in the box sees them. Here only `x0` is dynamic and it holds five positions, so the result has shape `(5, 2)`: one row of params for each blob. Inside a simulation these are the values of that simulation, so gradients pass through them.
- `self.gauss.brightness(x, y, params=p)` runs the method as a simulation of its own with the values `p`. One row has shape `(2,)`, which is a single blob, just as `Gaussian` was written to expect.

`vmap` then makes that call once for each row. The call is wrapped in a `lambda` so that `vmap` goes over the params and nothing else, here `x` and `y` are the same for every blob. (Writing `torch.vmap(self.gauss.brightness)(x, y, params=params)` would not do this, `torch.vmap` does not go over keyword arguments at all.)

The array from `get_values()` repeats a value that all the blobs share in every row. To avoid that, ask for a list and for the dimension to go over in each of its entries:

```python
values = self.gauss.get_values("list")  # one entry for each dynamic param
dims = self.gauss.get_dims()  # 0 to go over that entry, None for a shared value
batch_gauss = torch.vmap(lambda v: self.gauss.brightness(x, y, params=v), in_dims=(dims,))
return batch_gauss(values).sum(dim=0)
```

If `I0` were also dynamic and held a single value, the two lists would be `[x0, I0]` and `[0, None]`, and all five blobs would be handed the same `I0`.

The static params can be handled in the same way, with `get_values(dynamic=False)` and the `static_params` argument of the method. And none of this is special to hierarchical links, `params=` runs any `@forward` method of any module as a simulation of its own, also from inside a running simulation. What the link adds is the box in the graph and the meaning of the batch dimensions, which is explained below.

(Earlier versions of `caskade` passed these values to the method as arguments named `gauss_params` and `gauss_dims`. Those still work for now and give a `DeprecationWarning`.)

In [ ]:
params = Blobs.get_values()
print("notice params is still a 1d tensor, so we can use it like normal: ", params.shape)
plt.imshow(Blobs.brightness(X, Y, Blobs.get_values()))
plt.axis("off")
plt.show()

Yay! Our single Gaussian module has now become a flexible batched Gaussian module. What's more, this is fully integrated into the `caskade` system. The model is a function of a 1d vector of values and so we could run MCMC/optimization/etc on it. This also means that we can take advantage of regular `caskade` batching on top of this. Prepare to have you brain bend a little bit.

In [ ]:
# Let's give blobs a sigma parameter to control the gaussian blob it owns
# We make the Blobs sigma a batched param, so now the simulator is batched
#                                                      v: batch size of 6
Blobs.sigma = ck.Param("sigma", torch.linspace(0.5, 2, 6), shape=(), dynamic=True)
G.sigma = Blobs.sigma  # Set the gaussian blob sigma to point to Blobs sigma
params = Blobs.get_values()
print("Now the simulator is batched, the params are batched too: ", params.shape)
Blobs.graphviz()

Notice how params (or modules) in the batched section of the graph can refer to params (or modules) outside the batched section. This is because of some clever aspects of how `caskade` works. Try not to think about it too much, everything should work intuitively.

In [ ]:
results = torch.vmap(Blobs.brightness, in_dims=(None, None, 0))(X, Y, params)
fig, axarr = plt.subplots(1, params.shape[0], figsize=(5 * params.shape[0], 6))
for ax, res in zip(axarr, results):
    ax.imshow(res)
    ax.axis("off")
plt.show()

This is very neat! But we cheated a bit by batching a param in the main graph. Can we batch the `x0` param that is part of the hierarchical graph? Yes, to keep it straight, just think "outside-in", the first batch dimension will be for the main simulator and the second batch dimension will be for the hierarchical simulator.

In [ ]:
G.x0 = np.random.uniform(
    low=-4, high=4, size=(6, 5, 2)
)  # (main sim batch, hierarchical batch, x0 shape)
print(
    "The Gaussian module on its own doesn't know what all the batch dims are for, only in the context of the full simulator do they get their meaning:",
    G.x0.batch_shape,
)
print(
    "The Gaussian module mostly just knows that it should only ever actually see an x0 with this shape:",
    G.x0.shape,
)
params = Blobs.get_values()
print(
    "the 'params' has the same shape as before, just now instead of broadcasting the x0 shape it has a variety of values:",
    params.shape,
)
results = torch.vmap(Blobs.brightness, in_dims=(None, None, 0))(X, Y, params)
fig, axarr = plt.subplots(1, params.shape[0], figsize=(5 * params.shape[0], 6))
for ax, res in zip(axarr, results):
    ax.imshow(res)
    ax.axis("off")
plt.show()

### Which dimension is which?

Every example above follows one rule. Count the hierarchical links between a module and a param, these are the boxes around the param in the graph of that module, and call the number `d`. The module then reads the batch shape of the param as

```
(batch of the module ..., link 1, ..., link d)
```

- The last `d` batch dimensions are one for each hierarchical link, outermost first. They are part of a single simulation, so they are flattened into the 1D `params` vector. This is why `params` stayed 1D when `x0` held five positions.
- Any dimensions before those are a batch of whole simulations, used like any other batch dimension in `caskade` (here with `vmap` on the simulator). This is the `6` in the `(6, 5, 2)` example.
- A param with no batch dimensions has a single value that every copy shares, like `q` and `phi` above.
- A param that does have batch dimensions needs at least `d` of them. With fewer there is no telling which link they are for, so `caskade` raises an error.

The split is always made by the module you are talking to. `Blobs` has one link above `x0`, so the `5` is inside its simulation and the `6` is a batch. `G` has no link above `x0`, so on its own it sees both as ordinary batch dimensions, which is why only the full simulator can say what they are for.

Hierarchical links can be nested, each box adds one more dimension at the end. A param inside two boxes, in a simulator that is batched over 6, has batch shape `(6, n_outer, n_inner)`. The example below builds one.

One consequence is that a param inside the box cannot have a batch dimension for the main simulator alone, since its last batch dimension always belongs to the link. To change a value across the main batch while sharing it among the copies, point it at a param outside the box as we did with `sigma`, or repeat the value along the link dimension.

### Two levels

To see the rule at work, let's put a box inside a box. A `Cluster` is a group of blobs placed around a common centre, and a `Field` is several clusters in one image. Each level is written for a single case, and runs the level below with `vmap` over the values of that level.

In [ ]:
class Cluster(ck.Module):
    """A group of blobs placed around a common centre."""

    def __init__(self, gaussian, centre=None):
        super().__init__("cluster")
        self.hierarchical_link("gauss", gaussian)
        self.centre = ck.Param("centre", centre, shape=(2,))

    @ck.forward
    def brightness(self, x, y, centre):
        x, y = x - centre[0], y - centre[1]
        batch_gauss = torch.vmap(lambda p: self.gauss.brightness(x, y, params=p))
        return batch_gauss(self.gauss.get_values()).sum(dim=0)


class Field(ck.Module):
    """Several clusters in one image."""

    def __init__(self, cluster):
        super().__init__("field")
        self.hierarchical_link("cluster", cluster)

    @ck.forward
    def brightness(self, x, y):
        batch_cluster = torch.vmap(lambda p: self.cluster.brightness(x, y, params=p))
        return batch_cluster(self.cluster.get_values()).sum(dim=0)


n_clusters, n_blobs = 3, 4
G = Gaussian("G", q=1.0, phi=0.0, sigma=0.4)
# x0 is inside both boxes: (link from field, link from cluster, x0 shape)
G.x0 = 0.8 * torch.randn(n_clusters, n_blobs, 2)
# no batch dimensions: one value shared by all 12 blobs
G.I0.to_dynamic(1.0)
cluster = Cluster(G)
# centre is inside one box: (link from field, centre shape)
cluster.centre = torch.tensor([[-3.0, -2.0], [0.0, 3.0], [3.0, -1.0]])
field = Field(cluster)
field.graphviz()

In [ ]:
for module in (field, cluster, G):
    print(f"{module.name} params shape: {tuple(module.get_values().shape)}")

plt.imshow(field.brightness(X, Y, field.get_values()))
plt.axis("off")
plt.show()

The three modules read the same batch shapes, each from where it stands:

- `field` has two boxes around `x0` and one around `centre`. All of those dimensions are inside one simulation of the field, so its `params` is 1D with `3*4*2 + 1 + 3*2 = 31` values.
- `cluster` has one box around `x0` and none around `centre`. The leading `3` is a batch to it, so its `params` has shape `(3, 4*2 + 1 + 2)`. This is the array that `field` gets from `self.cluster.get_values()` and goes over with `vmap`. The shared `I0` is repeated in each row.
- `G` has no box around any of its params, so every dimension is a batch to it and its `params` has shape `(3, 4, 2 + 1)`. Inside a simulation of one cluster the leading `3` is gone, there `self.gauss.get_values()` has shape `(4, 3)`.

A batch of fields is one more dimension at the front, as before. An `x0` of shape `(6, 3, 4, 2)` gives `field` a `params` of shape `(6, 31)` to use with `vmap` on `field.brightness`.

What about a value that the blobs of a cluster share, but that differs between clusters, a common `sigma` for example? Giving `G.sigma` a batch shape of `(3,)` raises an error, since that param is inside two boxes and one dimension could be for either. Make it a param of `Cluster`, where it is inside one box, and point `G.sigma` at it, just as we did with `Blobs.sigma` above.

### Every value at call time: `static_params`

So far the parent handed the child its dynamic values, and the child read its static values from the graph as usual. That works as long as all the copies share the static values. But a static param may hold one value for each copy too, a fixed width for each blob for example. It is not in `params`, so `vmap` does not go over it, and every blob would be handed all five widths at once.

The `static_params` keyword closes this gap. It takes the static values in the layout that `get_values(dynamic=False)` returns, just as `params` takes the dynamic ones. With both keywords the call `self.gauss.brightness(x, y, params=p, static_params=s)` reads no value from the graph, everything it uses is one of its arguments.

In [ ]:
class MultiBlobAll(ck.Module):
    """Runs the blobs with every value passed in, dynamic and static."""

    def __init__(self, gaussian):
        super().__init__("blobs")
        self.hierarchical_link("gauss", gaussian)

    @ck.forward
    def brightness(self, x, y):
        params = self.gauss.get_values()  # dynamic values, one row for each blob
        static = self.gauss.get_values(dynamic=False)  # static values, one row for each blob
        blob = lambda p, s: self.gauss.brightness(x, y, params=p, static_params=s)
        return torch.vmap(blob)(params, static).sum(dim=0)


# sigma and I0 are static, with one value for each blob. q and phi are shared
G = Gaussian(
    "G", q=1.0, phi=0.0, sigma=torch.linspace(0.4, 1.2, 5), I0=torch.linspace(1.0, 0.2, 5)
)
G.x0 = torch.tensor([[0.0, 0.0], [3, 1], [-2, 4], [-3, -3], [1, -4]])  # dynamic
print("dynamic:", [p.name for p in G.dynamic_params], tuple(G.get_values().shape))
print("static: ", [p.name for p in G.static_params], tuple(G.get_values(dynamic=False).shape))

AllBlobs = MultiBlobAll(G)
plt.imshow(AllBlobs.brightness(X, Y, AllBlobs.get_values()))
plt.axis("off")
plt.show()

Each blob has its own width and brightness, and nothing had to be made dynamic for it. In the static array a shared value such as `q` is repeated in every row, as shared dynamic values are. To pass each value once, use `get_values("list", dynamic=False)` together with `get_dims(dynamic=False)`, as shown for the dynamic values above.

Having every value as an argument matters beyond this example. A call that reads nothing from the graph is an ordinary function of its arguments, and that is what the function transforms of `torch` and `jax` work on. `vmap` can go over any of the values, as it just did. Derivatives can be taken with respect to static values without making them dynamic first:

In [ ]:
p, s = G.get_values(), G.get_values(dynamic=False)  # one row for each blob


def light(static):  # total light of the first blob, as a function of its static values
    return G.brightness(X, Y, params=p[0], static_params=static).sum()


grad = torch.func.grad(light)(s[0])
for param, g in zip(G.static_params, grad):
    print(f"d(light)/d({param.name}) = {g.item():.2f}")

The same goes for compiling. A compiled function only notices that a value has changed when that value is one of its arguments, anything it reads from the graph is fixed at the time it is compiled. With the `jax` backend a method is compiled like this:

```python
blob = jax.jit(lambda x, y, p, s: G.brightness(x, y, params=p, static_params=s))
image = blob(X, Y, G.get_values(), G.get_values(dynamic=False))
```

Leave out `static_params` and the static values from the first call stay in the compiled function for good, whatever you set them to later. (Plain attributes of a module, such as a stored grid, are always fixed in this way.)

So inside a simulation you can always get to every value, `get_values()` and `get_values(dynamic=False)` on any module below, and hand them to any `@forward` method with `params` and `static_params`. That is enough to make `vmap` and `jit` work whatever the simulator looks like. One shortcut: a method that takes no argument other than its params needs no `vmap` at all, `self.gauss.method(params=params, static_params=static)` is run once for each row by `caskade` (see the batching section of the [advanced guide](AdvancedGuide.ipynb)).

## Going transdimensional, variable size batches inside the simulator

Sometimes when creating a forward model, we wish to specify population parameters and have the simulator create a sample from that population. It is possible to do this in `caskade`, though it does present some challenges given the typical workflow which involves specifying the shapes of all params in the simulation ahead of time. To make it work, we will have to construct the inputs to the batched simulator ourselves, inside the main simulator. The main tool that makes this feasible is the `dynamic_params` tuple, which gives us the order the params are expected in the simulator. Let's see how this works in a simple example that draws a population of gaussian blobs.

In [ ]:
class BlobPopulation(ck.Module):
    def __init__(
        self, gaussian, pop_x0_mean=None, pop_x0_std=None, pop_sigma_low=None, pop_sigma_high=None
    ):
        super().__init__()
        self.hierarchical_link("gauss", gaussian)
        self.pop_x0_mean = ck.Param("pop_x0_mean", pop_x0_mean, shape=(2,))
        self.pop_x0_std = ck.Param("pop_x0_std", pop_x0_std, shape=(2,))
        self.pop_sigma_low = ck.Param("pop_sigma_low", pop_sigma_low, shape=())
        self.pop_sigma_high = ck.Param("pop_sigma_high", pop_sigma_high, shape=())

    @ck.forward
    def brightness(self, x, y, N, pop_x0_mean, pop_x0_std, pop_sigma_low, pop_sigma_high):
        # Randomly draw a number of members in the population
        n = np.random.poisson(N)

        # Sample values for the params
        params = {}
        expand = torch.ones(size=(n, 2))
        params["x0"] = torch.normal(
            mean=pop_x0_mean.unsqueeze(0) * expand, std=pop_x0_std.unsqueeze(0) * expand
        )
        params["sigma"] = torch.rand(n) * (pop_sigma_high - pop_sigma_low) + pop_sigma_low

        # Create params list
        params = list(params[p.name] for p in self.gauss.dynamic_params)
        dims = [0, 0]

        batch_gauss = torch.vmap(self.gauss.brightness, in_dims=(None, None, dims))
        return batch_gauss(x, y, params).sum(dim=0)


G = Gaussian("G", q=1.0, phi=0.0, I0=1.0)
G.x0.to_dynamic([0, 0])  # set dummy values for the params
G.sigma.to_dynamic(0)
G.x0.batch_shape = (1,)  # set filler batch shape
G.sigma.batch_shape = (1,)

BlobPop = BlobPopulation(G, [0, 0], [1, 4], 0.1, 1)
BlobPop.to_dynamic()
BlobPop.graphviz()

In [ ]:
plt.imshow(BlobPop.brightness(X, Y, 20, BlobPop.get_values()))
plt.axis("off")
plt.show()

Now we can play around with the parameters and generate a new population!

In [ ]:
BlobPop.pop_x0_std = [3, 1]
BlobPop.pop_sigma_high = 0.5
plt.imshow(BlobPop.brightness(X, Y, 500, BlobPop.get_values()))
plt.axis("off")
plt.show()

And with that, you've simulated a population of gaussian blobs in an image. The `caskade` framework makes certain aspects of this kind of modelling very nice, mainly interacting with the population parameters in an intuitive/object-oriented way. Though there are some limitations. For example, such a simulator cannot be batched, since the variable size internal batch would conflict with any attempt to extend the tensors by adding an extra dimension. Further, the fact that you must construct the params yourself (inside the simulator) is somewhat fragile. Swapping out the Gaussian with a different type of blob model (say a Nuker profile) would require different inputs and so would crash the current setup; similarly changing which params of the Gaussian blob are dynamic would also crash the current version since it only generates a population with varying `x0` and `sigma`. Future updates may give a more robust framework for such a model, but for now we left the door open for users to have total flexibility in simulator construction.

As a tip for doing this kind of population modelling yourself, consider putting any metadata you need right alongside the params that need them. When you iterate over the `dynamic_params` tuple, it gives you access to the param objects and any meta data you wish to store, all in the correct order for passing to the hierarchical graph.

### We're doing it live

In [ ]:
class BlobPopulation_v2(ck.Module):
    def __init__(
        self, gaussian, pop_x0_mean=None, pop_x0_std=None, pop_sigma_low=None, pop_sigma_high=None
    ):
        super().__init__()
        self.gauss = gaussian
        self.pop_x0_mean = ck.Param("pop_x0_mean", pop_x0_mean, shape=(2,))
        self.pop_x0_std = ck.Param("pop_x0_std", pop_x0_std, shape=(2,))
        self.pop_sigma_low = ck.Param("pop_sigma_low", pop_sigma_low, shape=())
        self.pop_sigma_high = ck.Param("pop_sigma_high", pop_sigma_high, shape=())

    @ck.forward
    def sub_brightness(self, x, y, x0, sigma):
        self.gauss.x0 = x0
        self.gauss.sigma = sigma
        return self.gauss.brightness(x, y)

    @ck.forward
    def brightness(self, x, y, N, pop_x0_mean, pop_x0_std, pop_sigma_low, pop_sigma_high):
        # Randomly draw a number of members in the population
        n = np.random.poisson(N)

        # Sample values for the params
        expand = torch.ones(size=(n, 2))
        x0 = torch.normal(
            mean=pop_x0_mean.unsqueeze(0) * expand, std=pop_x0_std.unsqueeze(0) * expand
        )
        sigma = torch.rand(n) * (pop_sigma_high - pop_sigma_low) + pop_sigma_low

        batch_gauss = torch.vmap(self.sub_brightness, in_dims=(None, None, 0, 0))
        return batch_gauss(x, y, x0, sigma).sum(dim=0)


G = Gaussian("G", q=1.0, phi=0.0, I0=1.0)
G.x0.to_static()
G.sigma.to_static()

BlobPop = BlobPopulation_v2(G, [0, 0], [1, 4], 0.1, 1)
BlobPop.to_dynamic()
BlobPop.graphviz()

In [ ]:
plt.imshow(BlobPop.brightness(X, Y, 20, BlobPop.get_values()))
plt.axis("off")
plt.show()
BlobPop.pop_x0_std = [3, 1]
BlobPop.pop_sigma_high = 0.5
plt.imshow(BlobPop.brightness(X, Y, 500, BlobPop.get_values()))
plt.axis("off")
plt.show()

Whichever form is most intuitive to you should work fine. 

Note, you can make a hierarchical link if you like to the gaussian in order to have the graph print with a box around it. This doesn't affect the live params, but it is helpful for mentally keeping track of what's going on. 

### Nesting simulators, going it alone

`caskade` provides an array of tools for robustly and efficiently building complex simulators, including the hierarchical ones described above. However, sometimes for one reason or another, you might wish that you could make a hierarchical model where `caskade` won't try and do anything to help you and just let you control all the simulator interactions. This is a "meta link", and it is a way to hold one simulator as an attribute of another while having no `caskade` based interaction between them. The meta linked simulator wont even show up when you print the graph.

In [ ]:
class BlobPopulation_v3(ck.Module):
    def __init__(
        self, gaussian, pop_x0_mean=None, pop_x0_std=None, pop_sigma_low=None, pop_sigma_high=None
    ):
        super().__init__()
        self.meta.gauss = gaussian
        self.pop_x0_mean = ck.Param("pop_x0_mean", pop_x0_mean, shape=(2,))
        self.pop_x0_std = ck.Param("pop_x0_std", pop_x0_std, shape=(2,))
        self.pop_sigma_low = ck.Param("pop_sigma_low", pop_sigma_low, shape=())
        self.pop_sigma_high = ck.Param("pop_sigma_high", pop_sigma_high, shape=())

    @ck.forward
    def brightness(self, x, y, N, pop_x0_mean, pop_x0_std, pop_sigma_low, pop_sigma_high):
        # Randomly draw a number of members in the population
        n = np.random.poisson(N)

        # Sample values for the params
        params = {}
        expand = torch.ones(size=(n, 2))
        params["x0"] = torch.normal(
            mean=pop_x0_mean.unsqueeze(0) * expand, std=pop_x0_std.unsqueeze(0) * expand
        )
        params["sigma"] = torch.rand(n) * (pop_sigma_high - pop_sigma_low) + pop_sigma_low

        # Create params list
        params = list(params[p.name] for p in self.meta.gauss.dynamic_params)
        dims = [0, 0]

        batch_gauss = torch.vmap(self.meta.gauss.brightness, in_dims=(None, None, dims))
        return batch_gauss(x, y, params).sum(dim=0)


G = Gaussian("G", q=1.0, phi=0.0, I0=1.0)
G.x0.to_dynamic()
G.sigma.to_dynamic()

BlobPop = BlobPopulation_v3(G, [0, 0], [1, 4], 0.1, 1)
BlobPop.to_dynamic()
BlobPop.graphviz()

In [ ]:
plt.imshow(BlobPop.brightness(X, Y, 20, BlobPop.get_values()))
plt.axis("off")
plt.show()
BlobPop.pop_x0_std = [3, 1]
BlobPop.pop_sigma_high = 0.5
plt.imshow(BlobPop.brightness(X, Y, 500, BlobPop.get_values()))
plt.axis("off")
plt.show()

This version functions just as well as the others, so in some sense the choice is one of style and maintainability. If you make use of meta links too much, then you are back to regular object oriented programming and `caskade` isn't doing much for you. Still, the option is there if you need it.